#### Import Core Dependencies

In [13]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from typing import Any
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm.auto import tqdm
import ast
import statistics as stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# verify paths are operational
print(Path("../data/raw/").exists())
print(Path("../checkpoints").exists())

True
True


In [3]:
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA Available: False


### Load and Inspect Data

In [4]:
# create path variables for ease of navigation
DATA_PATH = Path("../data/raw/")

lines_path = DATA_PATH / "movie_lines.txt"
conversation_path = DATA_PATH / "movie_conversations.txt"

In [ ]:
# read first few lines
with open(lines_path, "r", encoding="iso-8859-1") as f:
    for _ in range(5):
        print(f.readline().strip())

L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!
L1044 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ They do to!
L985 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I hope so.
L984 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ She okay?
L925 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Let's go.


In [6]:
with open(conversation_path, "r", encoding="iso-8859-1") as f:
    for _ in range(5):
        print(f.readline().strip())

u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L198', 'L199']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L200', 'L201', 'L202', 'L203']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L204', 'L205', 'L206']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L207', 'L208']


In [7]:
with open(lines_path, "r", encoding="iso-8859-1") as f:
    num_lines = sum(1 for _ in f)

print(f"Movie lines: {num_lines:,}")

Movie lines: 304,713


In [ ]:
with open(conversation_path, "r", encoding="iso-8859-1") as f:
    num_conversations = sum(1 for _ in f)

print(f"Conversations: {num_conversations:,}")
# the documentation mentioned over 220K conversational exchanges which I presume would not whole conversations which is counted below

Conversations: 83,097


In [11]:
with open(
    lines_path,
    encoding="iso-8859-1"
) as f:

    sample_line = f.readline()

print(sample_line)
print(sample_line.split(" +++$+++ "))

L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!

['L1045', 'u0', 'm0', 'BIANCA', 'They do not!\n']


In [12]:
with open(
    conversation_path,
    encoding="iso-8859-1"
) as f:

    sample_conv = f.readline()

print(sample_conv)
print(sample_conv.split(" +++$+++ "))

u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']

['u0', 'u2', 'm0', "['L194', 'L195', 'L196', 'L197']\n"]


##### Initial Dataset Inspection

Initial inspection of the Cornell Movie-Dialogs Corpus revealed that the dataset is stored in a relational format rather than as fully assembled conversations. The `movie_lines.txt` file contains individual utterances associated with line identifiers, characters, and movies, while `movie_conversations.txt` contains ordered collections of line identifiers representing conversations between characters. Exploratory analysis identified approximately 304,713 individual movie lines and 83,097 conversations.

To better understand the conversational content and prepare the data for language modeling, the next step is to reconstruct complete conversations by mapping line identifiers to their corresponding utterances. This reconstruction process will produce a coherent dialogue corpus suitable for further exploratory analysis, tokenization, and Transformer training.

In [16]:
line_lookup = {}

with open(lines_path,"r", encoding="iso-8859-1") as f:

    for line in f:

        parts = line.strip().split(" +++$+++ ")

        line_id = parts[0]

        text = parts[-1]

        line_lookup[line_id] = text

print(
    f"Lines Loaded: {len(line_lookup):,}"
)

Lines Loaded: 304,713


In [17]:
sample_key = next(iter(line_lookup))

print(sample_key)

print(
    line_lookup[sample_key]
)

L1045
They do not!


In [20]:
conversations = []

with open(conversation_path, "r", encoding="iso-8859-1") as f:
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        line_ids = ast.literal_eval(parts[-1])
        
        # Builds the conversation
        dialogue = [line_lookup[line_id] for line_id in line_ids if line_id in line_lookup]
        
        if dialogue:
            conversations.append(dialogue)


print(
    f"Conversations Reconstructed: {len(conversations):,}"
)

Conversations Reconstructed: 83,097


In [31]:
sample_conversation = conversations[2]

for i, utterance in enumerate(
    sample_conversation,
    start=1
):
    print(
        f"{i}. {utterance}"
    )

1. No, no, it's my fault -- we didn't have a proper introduction ---
2. Cameron.
3. The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.
4. Seems like she could get a date easy enough...


In [34]:
for i in range(3):

    print(f"\nConversation {i+1}")
    print("-" * 40)

    for utterance in conversations[i]:
        print(utterance)



Conversation 1
----------------------------------------
Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.
Well, I thought we'd start with pronunciation, if that's okay with you.
Not the hacking and gagging and spitting part.  Please.
Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?

Conversation 2
----------------------------------------
You're asking me out.  That's so cute. What's your name again?
Forget it.

Conversation 3
----------------------------------------
No, no, it's my fault -- we didn't have a proper introduction ---
Cameron.
The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.
Seems like she could get a date easy enough...


##### Intial Data Preparation

- We will be adding boundary markers to our corpus so the Transformer model can learn turn behaviour

In [35]:
EOU_TOKEN = "<|eou|>"
EOD_TOKEN = "<|endofdialog|>"

In [37]:
processed_conversations = []

for conversation in conversations:

    dialogue_text = (
        f" {EOU_TOKEN} "
    ).join(conversation)

    dialogue_text += f" {EOD_TOKEN}"

    processed_conversations.append(
        dialogue_text
    )

In [41]:
processed_conversations[0:3]

["Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again. <|eou|> Well, I thought we'd start with pronunciation, if that's okay with you. <|eou|> Not the hacking and gagging and spitting part.  Please. <|eou|> Okay... then how 'bout we try out some French cuisine.  Saturday?  Night? <|endofdialog|>",
 "You're asking me out.  That's so cute. What's your name again? <|eou|> Forget it. <|endofdialog|>",
 "No, no, it's my fault -- we didn't have a proper introduction --- <|eou|> Cameron. <|eou|> The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does. <|eou|> Seems like she could get a date easy enough... <|endofdialog|>"]

In [42]:
word_counts = [len(conv.split()) for conv in processed_conversations]

print(f"Conversations : {len(processed_conversations):,}")
print(f"Total Words : {sum(word_counts):,}")

print(f"Min Words : {min(word_counts)}")
print(f"Mean Words : {stats.mean(word_counts):.2f}")
print(f"Median Words : {stats.median(word_counts)}")
print(f"Max Words : {max(word_counts)}")

Conversations : 83,097
Total Words : 3,515,653
Min Words : 4
Mean Words : 42.31
Median Words : 28
Max Words : 1245


In [ ]:
character_counts = [len(conv) for conv in processed_conversations] # each conv is a string now so we count each character like this

print(f"Mean Characters : {stats.mean(character_counts):.2f}")
print(f"Median Characters : {stats.median(character_counts)}")
print(f"Max Characters : {max(character_counts)}")

Mean Characters : 242.72
Median Characters : 167
Max Characters : 6678
